Acknowledgements

Name: Zuha Aqib, Farah Inayat, Zehra Ahmed   
Date: 11th June 2025   
ITA Assignment 5

In [1]:
from datetime import datetime   

# Capture start time
s_start_time = datetime.now()

print("Last time code executed:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

Last time code executed: 2025-06-11 17:55:41


# imports

In [2]:
# === INSTALL REQUIRED LIBRARIES (ONLY IN NOTEBOOKS) ===
%pip install transformers datasets accelerate peft bitsandbytes trl evaluate nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 26.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.3/366.3 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.7 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.6 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 8.0 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.3 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 70.7 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: nvidia

In [3]:
# === IMPORTS ===
import torch
import time
import os
import csv
from datetime import datetime
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from trl import DPOTrainer, DPOConfig
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction


2025-06-11 17:57:07.068854: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1749664627.278442      35 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1749664627.339525      35 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


# variables

In [4]:
# === DPO VARIABLES BLOCK ===

BASE_MODEL = "TinyLlama/TinyLlama_v1.1"
# Base pretrained model

LORA_MODEL_PATH = "/kaggle/input/best-lora-/other/default/1/kaggle/working/tinyllama-lora-sft"  # ← UPDATE THIS TO WHERE YOU SAVED LoRA MODEL
# Path to trained LoRA adapter

DATASET_NAME = "Intel/orca_dpo_pairs"
# Human preference dataset (prompt, chosen, rejected)

DPO_BETA = 0.1
# Preference strength. Try 0.05, 0.1, 0.3

DPO_LEARNING_RATE = 1e-5
# DPO training learning rate. Try 1e-5 to 5e-5

DPO_BATCH_SIZE = 4
# Per-device batch size

DPO_EPOCHS = 3
# Number of epochs for DPO training

DPO_GRAD_ACCUM = 4
# Gradient accumulation steps

MAX_LENGTH = 512
# Max token length for prompts

DPO_OUTPUT_DIR = "/kaggle/working/tinyllama-lora-dpo"
# Where to save the final DPO model

FP16 = False
USE_BF16 = False


# load lora

In [5]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, load_in_8bit=True, device_map="auto")
model = PeftModel.from_pretrained(base_model, LORA_MODEL_PATH)


tokenizer_config.json:   0%|          | 0.00/776 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/560 [00:00<?, ?B/s]

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


pytorch_model.bin:   0%|          | 0.00/4.40G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/129 [00:00<?, ?B/s]

# load data

In [6]:
from datasets import load_dataset

dpo_dataset = load_dataset(DATASET_NAME, split="train")
print(dpo_dataset.column_names)


README.md:   0%|          | 0.00/196 [00:00<?, ?B/s]

orca_rlhf.jsonl:   0%|          | 0.00/36.3M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/12859 [00:00<?, ? examples/s]

['system', 'question', 'chosen', 'rejected']


In [7]:
dpo_dataset = load_dataset(DATASET_NAME, split="train")

def format_for_dpo(example):
    return {
        "prompt": example["question"],
        "chosen": example["chosen"],
        "rejected": example["rejected"]
    }

num_samples = 5000
dpo_dataset = dpo_dataset.map(format_for_dpo)
dpo_dataset = dpo_dataset.shuffle(seed=42).select(range(num_samples))  # or fewer if needed


Map:   0%|          | 0/12859 [00:00<?, ? examples/s]

# train dpo

In [8]:
dpo_config = DPOConfig(
    beta=DPO_BETA,
    learning_rate=DPO_LEARNING_RATE,
    per_device_train_batch_size=DPO_BATCH_SIZE,
    num_train_epochs=DPO_EPOCHS,
    gradient_accumulation_steps=DPO_GRAD_ACCUM,
    max_length=MAX_LENGTH,
    output_dir=DPO_OUTPUT_DIR,
    logging_steps=10,
    save_strategy="epoch",
    report_to="none",
    #evaluation_strategy="no",
    fp16=FP16,
    bf16=USE_BF16
)

In [9]:
pip install --upgrade git+https://github.com/huggingface/trl.git

  Cloning https://github.com/huggingface/trl.git to /tmp/pip-req-build-xbu51hm5
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/trl.git /tmp/pip-req-build-xbu51hm5
  Resolved https://github.com/huggingface/trl.git to commit 1314aac502979d6ca7064e32e33b844e22f78823
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for trl: filename=trl-0.19.0-py3-none-any.whl size=365613 sha256=af602fd40da1dcf375529f3bdb1bab1a39b5d043aeb381e7a16a62a14acb53cd
  Stored in directory: /tmp/pip-ephem-wheel-cache-jzpgygtc/wheels/e7/e5/ec/8cce76372e10e954b47f3893a36ff7c0786d9dbc846efa8860
Successfully built trl
  Attempting uninstall: trl
    Found existing installation: trl 0.18.1
    Uninstalling trl-0.18.1:
      Successfully uninstalled trl-0.18.1
Note: you may need to restart the kernel to use updated packages.


In [10]:
!pip install -U trl

In [11]:
import trl
print(trl.__version__)

0.18.1


In [17]:
from trl import DPOTrainer, DPOConfig

trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=dpo_config,
    train_dataset=dpo_dataset,
    processing_class=tokenizer,
    #beta=DPO_BETA
)


No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [18]:
model.train()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(32000, 2048)
        (layers): ModuleList(
          (0-21): 22 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear8bitLt(
                (base_layer): Linear8bitLt(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): Linea

In [19]:
for name, param in model.named_parameters():
    if param.requires_grad:
        print(f"{name} requires grad")


base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight requires grad
base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight requires grad
base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight requires grad
base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight requires grad
base_model.model.model.layers.1.self_attn.q_proj.lora_A.default.weight requires grad
base_model.model.model.layers.1.self_attn.q_proj.lora_B.default.weight requires grad
base_model.model.model.layers.1.self_attn.v_proj.lora_A.default.weight requires grad
base_model.model.model.layers.1.self_attn.v_proj.lora_B.default.weight requires grad
base_model.model.model.layers.2.self_attn.q_proj.lora_A.default.weight requires grad
base_model.model.model.layers.2.self_attn.q_proj.lora_B.default.weight requires grad
base_model.model.model.layers.2.self_attn.v_proj.lora_A.default.weight requires grad
base_model.model.model.layers.2.self_attn.v_proj.lora_B.default.w

In [20]:
start_time = datetime.now()
print("Start time:", start_time.strftime('%Y-%m-%d %H:%M:%S'))


Start time: 2025-06-11 18:08:46


In [21]:
trainer.train()

Step,Training Loss
10,0.691200
20,0.673600
30,0.664800
40,2.966900
50,12.710800
60,10.815800
70,6.995000
80,8.843900
90,8.069100
100,6.433100


TrainOutput(global_step=936, training_loss=8.610595990449955, metrics={'train_runtime': 7887.0136, 'train_samples_per_second': 1.902, 'train_steps_per_second': 0.119, 'total_flos': 0.0, 'train_loss': 8.610595990449955, 'epoch': 2.992})

In [22]:
end_time = datetime.now()
duration = end_time - start_time
training_minutes = round(duration.total_seconds() / 60, 2)

In [23]:
model.save_pretrained(DPO_OUTPUT_DIR)
tokenizer.save_pretrained(DPO_OUTPUT_DIR)

('/kaggle/working/tinyllama-lora-dpo/tokenizer_config.json',
 '/kaggle/working/tinyllama-lora-dpo/special_tokens_map.json',
 '/kaggle/working/tinyllama-lora-dpo/tokenizer.model',
 '/kaggle/working/tinyllama-lora-dpo/added_tokens.json',
 '/kaggle/working/tinyllama-lora-dpo/tokenizer.json')

model.save_pretrained(DPO_OUTPUT_DIR)
tokenizer.save_pretrained(DPO_OUTPUT_DIR)


# eval

In [24]:
eval_prompts = [
    "Explain the difference between machine learning and deep learning.",
    "What is the Pythagorean theorem used for?",
    "List three causes of World War II.",
    "Translate 'Good morning' into French.",
    "What are some benefits of daily exercise?",
    "Summarize the plot of Romeo and Juliet.",
    "What does HTTP stand for?",
    "Give an example of a palindrome.",
    "How do you boil an egg perfectly?",
    "Describe the lifecycle of a butterfly."
]

reference_answers = [
    "Machine learning is a broader concept of algorithms that learn from data. Deep learning is a subset of machine learning that uses neural networks with multiple layers.",
    "The Pythagorean theorem helps calculate the length of a side in a right triangle: a² + b² = c².",
    "Three causes of WWII include the Treaty of Versailles, the rise of fascism, and the invasion of Poland by Nazi Germany.",
    "‘Good morning’ in French is ‘Bonjour’.",
    "Benefits of daily exercise include improved mood, better sleep, and stronger muscles.",
    "Romeo and Juliet is a tragedy about two young lovers from feuding families who ultimately die because of misunderstandings and conflict.",
    "HTTP stands for HyperText Transfer Protocol.",
    "A palindrome is a word like 'racecar' that reads the same backward and forward.",
    "Boil an egg by placing it in boiling water for 9-12 minutes depending on the desired hardness.",
    "A butterfly’s lifecycle includes egg, larva (caterpillar), pupa (chrysalis), and adult stages."
]

In [25]:
smoother = SmoothingFunction().method2

def generate_response(prompt, model, tokenizer):
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            input_ids=input_ids,
            max_new_tokens=100,
            temperature=0.7,
            top_p=0.9
        )
    return tokenizer.decode(output_ids[0], skip_special_tokens=True).split("### Response:")[-1].strip()

In [26]:
def calculate_bleu_scores(preds, refs):
    scores = []
    for pred, ref in zip(preds, refs):
        score = sentence_bleu([ref.split()], pred.split(), smoothing_function=smoother)
        scores.append(score)
    return scores

In [27]:
model.eval()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(32000, 2048)
        (layers): ModuleList(
          (0-21): 22 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear8bitLt(
                (base_layer): Linear8bitLt(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): Linea

In [28]:
# Run evaluation
dpo_outputs = [generate_response(p, model, tokenizer) for p in eval_prompts]
bleu_dpo_scores = calculate_bleu_scores(dpo_outputs, reference_answers)
bleu_dpo_avg = round(sum(bleu_dpo_scores) / len(bleu_dpo_scores), 4)

print("DPO BLEU Score:", bleu_dpo_avg)

/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(


DPO BLEU Score: 0.0262


# csv

In [29]:
# Format log
log = {
    "timestamp": start_time.strftime('%Y-%m-%d %H:%M:%S'),
    "training_time": round(duration.total_seconds() / 60, 2),
    "base_model": BASE_MODEL,
    "lora_model_path": LORA_MODEL_PATH,
    "dataset": DATASET_NAME,
    "num_samples": num_samples,
    "dpo_beta": DPO_BETA,
    "dpo_learning_rate": DPO_LEARNING_RATE,
    "dpo_batch_size": DPO_BATCH_SIZE,
    "dpo_epochs": DPO_EPOCHS,
    "dpo_grad_accum": DPO_GRAD_ACCUM,
    "max_length": MAX_LENGTH,
    "bleu_dpo_avg": bleu_dpo_avg
}

for i in range(10):
    log[f"q{i+1}_output"] = dpo_outputs[i]
    log[f"bleu{i+1}"] = round(bleu_dpo_scores[i], 4)

# Save to CSV
csv_path = "/kaggle/working/dpo_experiments_log.csv"
file_exists = os.path.isfile(csv_path)

with open(csv_path, mode="a", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=log.keys())
    if not file_exists:
        writer.writeheader()
    writer.writerow(log)